DQN
input: len 4 vec - pos + loot_pos.
output: len 4 action vec, prob 0-1 valued. (Soft max activation function)

Middle layers: 3 hidden linear layers, choice activation function carefully

initialize
    Setup env + agent
    Setup start pos + loot pos
Run game
    Send state vector to network
    Find action and act
update weights
    Last moves are most important - Use discount factor


In [22]:
#DQN Model
import numpy as np

In [23]:
class Actions:
    def __init__(self):
        self.MOVE_UP = 0
        self.MOVE_DOWN = 1
        self.MOVE_LEFT = 2
        self.MOVE_RIGHT = 3
        self.available_actions = [self.MOVE_UP, self.MOVE_DOWN, self.MOVE_LEFT, self.MOVE_RIGHT]

class Position:
    def __init__(self, x, y):
        self.x = int(x)
        self.y = int(y)

class Agent:
    def __init__(self, name, position, env):
        self.name = name
        self.position = position
        self.env = env
        self.action_list = Actions().available_actions
        self.history = []

    def move(self, state_index, agent_action):
        """Move the agent in the environment based on the action"""
        actions = [Position(0,1), Position(0,-1), Position(-1,0), Position(1,0)]
        action = actions[agent_action]
        new_position = Position(self.position.x + action.x, self.position.y + action.y)

        if self.env.valid_state(new_position):
            self.position = new_position
        else: #If the move is invalid, the agent stays in the same position
            self.position = self.position
            print("Invalid move")

    def pick_action(self, state_index, epsilon_greedy=0.2):
        #Call model to pick action
        return self.state_action_values[state_index].argmax()
        #Return a action vector with probabilities for a given action with respect to the input state.

In [24]:
class Environment:
    def __init__(self, size=(10.0, 10.0)):
        self.size = size
        self.starting_positions = []  # List to store the initial positions of agents
        self.agents = []
        self.loot = Position(0, 0)
        self.winners = []
        self.game_index = 0
        self.get_list_of_states()
        self.number_of_moves_in_game = 0
        self.length_of_games = []
        self.history = []

    def set_learning_parameters(self, q_learning_rate, learning_length, move_loot):
        self.q_learning_rate = q_learning_rate
        self.learning_length = learning_length
        self.move_loot = move_loot

    def get_list_of_states(self):        
        self.list_of_states = []
        if self.move_loot == True:
            for i in range(self.size[0]):
                for j in range(self.size[1]):
                    for n in range(self.size[0]):
                        for m in range(self.size[1]):
                            self.list_of_states.append((i, j, n, m))
        else:
            for i in range(self.size[0]):
                for j in range(self.size[1]):
                    self.list_of_states.append((i, j))
        self.list_of_state = np.array(self.list_of_states)

    def valid_state(self, position):
        return 0 <= position.x < self.size[0] and 0 <= position.y < self.size[1]

    def add_agent(self, agent):
        self.agents.append(agent)
        self.starting_positions.append(agent.position)  # Store the starting position of the agent

    def reset(self):
        # set new random positions for the agents
        for agent, position in zip(self.agents, self.starting_positions):
            agent.position = position
        print(position.x, position.y)
        print(self.loot.x, self.loot.y)
        self.length_of_games.append(self.number_of_moves_in_game)
        self.number_of_moves_in_game = 0

    def move_loot(self):
        self.loot.x = rn.randint(0, self.size[0]-1)
        self.loot.y = rn.randint(0, self.size[1]-1)

    def get_state(self):
        # return the state of the environment in Position format
        state = []
        for agent in self.agents:
            state.append(agent.position.x)
            state.append(agent.position.y)
        if self.move_loot == True:
            state.append(self.loot.x)
            state.append(self.loot.y)
        state_index = (self.list_of_states == np.array(state)).sum(axis=1).argmax()  #TODO: faster
        return state, state_index

    def act(self, state_index, agent_actions, verbose=True):
        # Update the agents' positions
        for agent, agent_actions in zip(self.agents, agent_actions):
            if agent.is_alive() == True:
                agent.move(state_index, agent_actions)
                self.number_of_moves_in_game += 1
                self.history.append((state_index, agent_actions))

                # Check if the agent has reached the loot
                if agent.position.x == self.loot.x and agent.position.y == self.loot.y:
                    self.winners.append(agent)
                    self.game_index += 1
                    if verbose == True:
                        print(f'Game {self.game_index} won by {agent.name} in {self.number_of_moves_in_game}!')
                    self.reset()
                    if self.move_loot == True:
                        self.move_loot()
                    agent.update_q(1, self.q_learning_rate, self.learning_length)
                    for agent_ in self.agents:
                        agent_.history = []

In [26]:
import numpy as np
import matplotlib.pyplot as plt

env = Environment(size=(10, 10))
env.set_learning_parameters(q_learning_rate=1.4, learning_length=20, move_loot=True)

player_one = Agent('Alice', Position(9, 9), env=env)

env.add_agent(player_one)

Idea: Run game until an agent wins.
Then take the trajectory, remove all but the last 15 steps, add random noise to the rest, use as truth value for optimizing.
Or use a homogenous truth function for all but the last 15 steps in order to remove random bias.

Or introduce a distance to loot as a running reward measure. - Feels like cheating and will direct the network along forcefull trajectories.

From CHATGPT script vs. own script: 
Own script with learning length has a tendency to learn the wrong thing, and converge on a single action, despite different 

Benchmark against morte carlo statistics on distance between 2 random points in 2d grid. How long does it take to reach 90% performance compared to optimal. Make graph with training speeds for Q-learning and DQN with various grid sizes. Showing it takes much longer to compute Q-learning for larger games.

Use agent vs agent with Q-learning vs DQN decision making processes, plot wins pr 100 games (where win is the one to achieve loot first) and show that one agent is superior when complexity increases. Note that Q learning might be faster for small simple games.

Create visualizations for optimal policy for static loot and single trajectories for moveable loot.

Reduce games to less than 100 games and discard that memory if steps run out.

Compare the Q-learning matrix with the derived DQN representation of the Q-matrix. How fast is convergence and where is the different method most effective?

How does each method capture dynamics on the "smaller" side of the loot?

What is the optimal length of maximum amouth of steps pr game and the reward system, as it should wary with different game grid sizes.

Make a new game with 2 agents, one act as loot and one act as adventurer, see what dynamics will unfold and learn how to descripe them.
Or! one acts as a strong impassable 

Streamlit PHYSICS for optimization processes.
PPO can be imported from stablebaselines.

Also include PPO/stable baselines for comparision in the above mentioned plots.

Next step is continous state spaces.